In [1]:
%run ../cpch_source.py

In [2]:
rpy2.robjects.numpy2ri.activate()

In [3]:
#install adaFilter to you can use it in Python
#un-comment out commented lines when running for first time
#utils = rpackages.importr('utils')
#utils.chooseCRANmirror(ind=1)
packnames = ('devtools')
#utils.install_packages(StrVector(packnames))
devtools = importr('devtools')
install_git = devtools.install_github
#install_git("jingshuw/adaFilter", force = True)
adaFilter = importr('adaFilter')
af = adaFilter.adaFilter
classical = adaFilter.ClassicalMTP

In [4]:
#change directory to where the dmd data output from DMD_data_final.R is saved
dmd = pd.read_csv('dmd_avging.csv') 

In [5]:
np.random.seed(211)

In [35]:
def run_followup(data, og_study_num, alpha):
    '''
    og_study_num: 1, 2, 3, or 4
    '''
    pvals = ['pvalue.1', 'pvalue.2', 'pvalue.3', 'pvalue.4']
    ts = ['ts.1', 'ts.2', 'ts.3', 'ts.4']
    pval_str = 'pvalue.' + str(og_study_num)
    #perform BH on chosen "original" study
    rej = mt.multipletests(data[pval_str], alpha, method = 'fdr_bh')[0]
    data_follow = data[rej]
    print(data_follow.shape[0])
    #pick out p-values from remaining studies
    pvals.remove(pval_str)
    XX_pvals =  np.array(data_follow[pvals])
    ts.remove('ts.' + str(og_study_num))
    XX = np.array(data_follow[ts])
    #store num rejections for each value of r
    cpch_fisher_storey, cpch_fisher_bh = [], []
    fisher_storey, fisher_bh, fisher_dhh, fisher_sdhh = [], [], [], []
    adafilter = []
    for r in np.arange(2, 4, 1):
        #cPCH-based methods
        cpch_f = cpch(XX, 3, r, f_fisher, norm.pdf, norm.cdf, truncnorm.rvs, 100000)
        cpch_fisher_bh.append(sum(mt.multipletests(cpch_f, alpha, method = 'fdr_bh')[0]))
        cpch_fisher_storey.append(sum(storey(cpch_f, alpha)))

        #fisher-based methods
        nr, nc = XX_pvals.shape
        Br = ro.r.matrix(XX_pvals, nrow=nr, ncol=nc)
        fisher_results = classical(Br, r = r, alpha = alpha, method = "Fisher")
        fisher_pvals = np.array(list(pd.DataFrame(fisher_results).to_numpy()[1, :]))
        s_tau = fisher_pvals[fisher_pvals <= 0.1]
        dhh_pvals = s_tau/0.1
        fisher_dhh.append(sum(mt.multipletests(dhh_pvals, alpha, method = 'fdr_bh')[0]))
        fisher_bh.append(sum(mt.multipletests(fisher_pvals, alpha, method = 'fdr_bh')[0]))
        fisher_storey.append(sum(storey(fisher_pvals, alpha)))
        fisher_sdhh.append(sum(storey(dhh_pvals, 0.05)))
        
        #adafilter
        ada_results = af(Br, r = r, alpha = alpha)
        ada_decision = pd.DataFrame(ada_results).to_numpy()[:,0]
        adafilter.append(sum(ada_decision))
        
    rejections = [cpch_fisher_storey, cpch_fisher_bh, fisher_storey, fisher_bh, fisher_dhh, fisher_sdhh, adafilter]
    print("cPCH-Fisher-Storey & " + str(rejections[0][0]) + " & " + str(rejections[0][1]) + ' \\\\')
    print("cPCH-Fisher-BH & " + str(rejections[1][0]) + " & " + str(rejections[1][1]) + ' \\\\')
    print("Fisher-Storey & " + str(rejections[2][0]) + " & " + str(rejections[2][1]) + ' \\\\')
    print("Fisher-BH & " + str(rejections[3][0]) + " & " + str(rejections[3][1]) + ' \\\\')
    print("Fisher-DHH & " + str(rejections[4][0]) + " & " + str(rejections[4][1]) + ' \\\\')
    print("Fisher-sDHH & " + str(rejections[5][0]) + " & " + str(rejections[5][1]) + ' \\\\')
    print("adaFilter & " + str(rejections[6][0]) + " & " + str(rejections[6][1]) + ' \\\\')

    return(rejections)

In [36]:
rejection_1 = run_followup(dmd, 1, 0.1)

73


/Users/Biyonka/Documents/Research/minmax/cpch/cpch_source.py:172: RuntimeWarning: divide by zero encountered in log
  


cPCH-Fisher-Storey & 73 & 69 \\
cPCH-Fisher-BH & 64 & 53 \\
Fisher-Storey & 71 & 63 \\
Fisher-BH & 64 & 54 \\
Fisher-DHH & 60 & 28 \\
Fisher-sDHH & 64 & 38 \\
adaFilter & 64.0 & 56.0 \\


In [37]:
rejection_2 = run_followup(dmd, 2, 0.1)

437
cPCH-Fisher-Storey & 356 & 95 \\
cPCH-Fisher-BH & 243 & 68 \\
Fisher-Storey & 305 & 51 \\
Fisher-BH & 227 & 39 \\
Fisher-DHH & 125 & 12 \\
Fisher-sDHH & 148 & 11 \\
adaFilter & 221.0 & 76.0 \\


In [38]:
rejection_3 = run_followup(dmd, 3, 0.1)

246
cPCH-Fisher-Storey & 228 & 82 \\
cPCH-Fisher-BH & 173 & 59 \\
Fisher-Storey & 219 & 65 \\
Fisher-BH & 166 & 57 \\
Fisher-DHH & 117 & 35 \\
Fisher-sDHH & 155 & 37 \\
adaFilter & 164.0 & 65.0 \\


In [39]:
rejection_4 = run_followup(dmd, 4, 0.1)

588
cPCH-Fisher-Storey & 360 & 68 \\
cPCH-Fisher-BH & 237 & 54 \\
Fisher-Storey & 286 & 31 \\
Fisher-BH & 204 & 31 \\
Fisher-DHH & 122 & 9 \\
Fisher-sDHH & 131 & 9 \\
adaFilter & 223.0 & 78.0 \\


In [40]:
mean_rej = np.mean(np.array([rejection_1, rejection_2, rejection_3, rejection_4]), axis = 0)
mean_rej

array([[254.25,  78.5 ],
       [179.25,  58.5 ],
       [220.25,  52.5 ],
       [165.25,  45.25],
       [106.  ,  21.  ],
       [124.5 ,  23.75],
       [168.  ,  68.75]])

In [41]:
#get avg number of rejections across all 4 pseudo-studies
mean_rejections = np.mean(np.array([rejection_1, rejection_2, rejection_3, rejection_4]), axis = 0)
print('r = 2 & ' + str(mean_rej[0][0]) + " & " + str(mean_rej[1][0]) + " & " + str(mean_rej[2][0]) + " & " + str(mean_rej[3][0])  + " & " + str(mean_rej[4][0])  + " & " + str(mean_rej[5][0])  + " & " + str(mean_rej[6][0])  + ' \\\\')
print('r = 3 & ' + str(mean_rej[0][1]) + " & " + str(mean_rej[1][1]) + " & " + str(mean_rej[2][1]) + " & " + str(mean_rej[3][1])  + " & " + str(mean_rej[4][1])  + " & " + str(mean_rej[5][1])  + " & " + str(mean_rej[6][1])  + ' \\\\')


r = 2 & 254.25 & 179.25 & 220.25 & 165.25 & 106.0 & 124.5 & 168.0 \\
r = 3 & 78.5 & 58.5 & 52.5 & 45.25 & 21.0 & 23.75 & 68.75 \\


In [42]:
# #get list of genes that were selected by cPCH using study 3
def get_cpch_rejections(og_study_num, alpha): 
    r = 3
    pvals = ['pvalue.1', 'pvalue.2', 'pvalue.3', 'pvalue.4']
    ts = ['ts.1', 'ts.2', 'ts.3', 'ts.4']
    pval_str = 'pvalue.' + str(og_study_num)
    #perform BH on chosen "original" study
    rej = mt.multipletests(dmd[pval_str], alpha, method = 'fdr_bh')[0]
    data_follow = dmd[rej]
    #pick out p-values from remaining studies
    pvals.remove(pval_str)
    XX_pvals =  np.array(data_follow[pvals])
    ts.remove('ts.' + str(og_study_num))
    XX = np.array(data_follow[ts])
    #cpch-based methods
    cpch_f = cpch(XX, 3, 3, f_fisher, norm.pdf, norm.cdf, truncnorm.rvs, 100000)
    data_follow['cpch_f_s_rej'] = storey(cpch_f, alpha)
    data_follow['cpchfpval'] = cpch_f
    rej_genes = data_follow[data_follow['cpch_f_s_rej'] == True].sort_values('Gene.symbol')['Gene.symbol']
    return rej_genes, data_follow

In [43]:
rej_genes_1, data_follow_1 = get_cpch_rejections(1, 0.1)
rej_genes_2, data_follow_2 = get_cpch_rejections(2, 0.1)
rej_genes_3, data_follow_3 = get_cpch_rejections(3, 0.1)
rej_genes_4, data_follow_4 = get_cpch_rejections(4, 0.1)

/Users/Biyonka/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/Users/Biyonka/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/Users/Biyonka/Documents/Research/minmax/cpch/cpch_source.py:172: RuntimeWarning: divide by zero encountered in log
  
/Users/Biyonka/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using 

In [44]:
#find genes rejected by all 4 pseudo-follow-up studies 
from functools import reduce
genes = reduce(np.intersect1d, (rej_genes_1, rej_genes_2, rej_genes_3, rej_genes_4))

In [45]:
cpch_rej_3_full = data_follow_3[data_follow_3['cpch_f_s_rej'] == True].sort_values('Gene.symbol')
cpch_rej_3 = cpch_rej_3_full.sort_values('cpchfpval').iloc[:20,:]

In [46]:
#download gene annotations that tell you about gene function
#set to directory where gene annotations are located
gpl246 = pd.read_csv("GPL246_annot.csv", sep = "\t")
gpl8300 = pd.read_csv("GPL8300_annot.csv", sep = "\t")
gpl96 = pd.read_csv("GPL96_annot.csv", sep = "\t")
#gpl246 = gpl246.sort_values('Gene symbol')
final_gene_data = pd.concat([gpl96, gpl246, gpl8300], axis = 0).sort_values('Gene symbol')
# gpl96 <- gpl96[order(gpl96.ID), ]
selected_genes_3 = final_gene_data[final_gene_data['Gene symbol'].isin(cpch_rej_3['Gene.symbol'])].drop_duplicates(subset=['Gene symbol'])
selected_genes_3 = selected_genes_3.merge(cpch_rej_3, left_on ='Gene symbol', right_on = 'Gene.symbol' )
selected_genes_3 = selected_genes_3[['Gene symbol', 'cpchfpval', 'GO:Process']].sort_values('Gene symbol')
selected_genes_3.columns = ['genesymbol', 'cpchfpval', 'function']
selected_genes_3['cpchfpval'] = np.round(selected_genes_3['cpchfpval'], 5)

In [47]:
def polish_annotations(annotation):
    fxn = annotation.split('///')[:2]
    return ('/'.join(fxn))

polish_annotations = np.vectorize(polish_annotations)
new_annotations = polish_annotations(np.array(selected_genes_3['function']))
selected_genes_3['function'] = new_annotations
#set to directory where gene annotations are located
selected_genes_3.to_csv('follow_up_3_short.csv', index= False)

In [48]:
selected_genes_3

,genesymbol,cpchfpval,function
0,ART3,0.00011,protein ADP-ribosylation
1,CFHR1,0.00075,complement activation
2,CHRNA1,0.00002,cation transmembrane transport/muscle cell cel...
3,DAB2,0.00001,Wnt signaling pathway/apoptotic process
4,EEF1A1,0.00002,cellular response to epidermal growth factor s...
5,EZR,0.00003,actin cytoskeleton reorganization/actin filame...
6,HLA-DMB,0.00045,MHC class II protein complex assembly/antigen ...
7,HLA-DRA,0.00062,T cell costimulation/T cell receptor signaling...
8,LAMB2,0.00034,Schwann cell development/astrocyte development
9,LAPTM5,0.00005,transport


In [49]:
def run_followup_2study_filter(data, og_study_nums, alpha):
    '''
    og_study_nums: list of 2 values in [1, 2, 3, 4]
    '''
    pvals = ['pvalue.1', 'pvalue.2', 'pvalue.3', 'pvalue.4']
    ts = ['ts.1', 'ts.2', 'ts.3', 'ts.4']
    pval_str_1 = 'pvalue.' + str(og_study_nums[0])
    pval_str_2 = 'pvalue.' + str(og_study_nums[1])
    data_filtered = data[[pval_str_1, pval_str_2]]
    #perform maxp on filtered data
    nr, nc = np.array(data_filtered).shape
    Br = ro.r.matrix(np.array(data_filtered), nrow=nr, ncol=nc)
    fisher_results = classical(Br, r = 2, alpha = alpha, method = "Fisher")
    fisher_rejections = np.array(list(pd.DataFrame(fisher_results).to_numpy()[0, :]))
    data_follow = data[fisher_rejections]
    print(data_follow.shape[0])

    #pick out p-values from remaining studies
    pvals.remove(pval_str_1)
    pvals.remove(pval_str_2)
    XX_pvals =  np.array(data_follow[pvals])
    ts.remove('ts.' + str(og_study_nums[0]))
    ts.remove('ts.' + str(og_study_nums[1]))
    XX = np.array(data_follow[ts])
    r = 2
    #cpch-based methods
    cpch_f = cpch(XX, 2, r, f_fisher, norm.pdf, norm.cdf, truncnorm.rvs, 100000)
    cpch_fisher_bh = (sum(mt.multipletests(cpch_f, alpha, method = 'fdr_bh')[0]))
    cpch_fisher_storey = (sum(storey(cpch_f, alpha)))

    #fisher-based methods
    nr, nc = XX_pvals.shape
    Br = ro.r.matrix(XX_pvals, nrow=nr, ncol=nc)
    fisher_results = classical(Br, r = 2, alpha = alpha, method = "Fisher")
    fisher_pvals = np.array(list(pd.DataFrame(fisher_results).to_numpy()[1, :]))
    s_tau = fisher_pvals[fisher_pvals <= 0.1]
    dhh_pvals = s_tau/0.1
    fisher_dhh = (sum(mt.multipletests(dhh_pvals, alpha, method = 'fdr_bh')[0]))
    fisher_bh = (sum(mt.multipletests(fisher_pvals, alpha, method = 'fdr_bh')[0]))
    fisher_storey = (sum(storey(fisher_pvals, alpha)))
    fisher_sdhh = (sum(storey(dhh_pvals, 0.05)))

    #adafilter
    ada_results = af(Br, r = 2, alpha = alpha)
    ada_decision = pd.DataFrame(ada_results).to_numpy()[:,0]
    adafilter = (sum(ada_decision))

    rejections = [cpch_fisher_storey, cpch_fisher_bh, fisher_storey, fisher_bh, fisher_dhh, fisher_sdhh, int(adafilter)]

    return(rejections)

In [50]:
studies = [1, 2, 3, 4]
combinations = list(itertools.combinations(studies, 2))
follow2 = []
for comb in combinations:
     follow2.append(run_followup_2study_filter(dmd, comb, 0.05))

25
9
29
47
208
112


In [51]:
mean_rej_2 = np.round(np.mean(np.array(follow2), axis = 0), 2)
print('r = 2 & ' + str(mean_rej_2[0]) + " & " + str(mean_rej_2[1]) + " & " + str(mean_rej_2[2]) + " & " + str(mean_rej_2[3])  + " & " + str(mean_rej_2[4])  + " & " + str(mean_rej_2[5])  + " & " + str(mean_rej_2[6])  + ' \\\\')


r = 2 & 33.5 & 26.5 & 31.67 & 23.67 & 13.67 & 18.5 & 26.5 \\


In [52]:
studies = [1, 2, 3, 4]
combinations = list(itertools.combinations(studies, 2))
follow2 = []
for comb in combinations:
     follow2.append(run_followup_2study_filter(dmd, comb, 0.1))

51
12
44
94
272
163


## Average Power

In [6]:
#compute p-values for r = 2, 3, 4 across all 4 studies
def calc_avg_power(XX, XX_pvals, n, alpha):
    avg_pow_cpch_f, avg_pow_cpch_s, avg_pow_cpch_b = [], [], []
    avg_pow_f, avg_pow_s, avg_pow_b = [], [], []
    se_cpch_f, se_cpch_s, se_cpch_b = [], [], []
    se_f, se_s, se_b = [], [], []
    for r in np.arange(2, n+1, 1):
        cpch_f = cpch(XX, n, r, f_fisher, norm.pdf, norm.cdf, truncnorm.rvs, 10000)
        cpch_s = cpch(XX, n, r, f_simes, norm.pdf, norm.cdf, truncnorm.rvs, 10000)
        cpch_b = cpch(XX, n, r, f_bonferroni, norm.pdf, norm.cdf, truncnorm.rvs, 10000)
        #look at what is rejected at nominal level 0.05
        avg_pow_cpch_f.append(np.mean(cpch_f <= alpha))
        avg_pow_cpch_s.append(np.mean(cpch_s <= alpha))
        avg_pow_cpch_b.append(np.mean(cpch_b <= alpha))
        se_cpch_f.append(np.std(cpch_f <= alpha,  ddof = 1)/np.sqrt(len(cpch_f)))
        se_cpch_s.append(np.std(cpch_s <= alpha,  ddof = 1)/np.sqrt(len(cpch_s)))
        se_cpch_b.append(np.std(cpch_b <= alpha,  ddof = 1)/np.sqrt(len(cpch_b)))
        #fisher-based methods
        nr, nc = XX_pvals.shape
        Br = ro.r.matrix(XX_pvals, nrow=nr, ncol=nc)
        fisher_results = classical(Br, r, alpha = alpha, method = "Fisher")
        fisher_pvals = np.array(list(pd.DataFrame(fisher_results).to_numpy()[1, :]))
        simes_results = classical(Br, r, alpha = alpha, method = "Simes")
        simes_pvals = np.array(list(pd.DataFrame(simes_results).to_numpy()[1, :]))
        bon_results = classical(Br, r, alpha = alpha, method = "Bonferroni")
        bon_pvals = np.array(list(pd.DataFrame(bon_results).to_numpy()[1, :]))
        #look at what is rejected at nominal level 0.05
        avg_pow_f.append(np.mean(fisher_pvals <= alpha))
        avg_pow_s.append(np.mean(simes_pvals <= alpha))
        avg_pow_b.append(np.mean(bon_pvals <= alpha))
        se_f.append(np.std(fisher_pvals <= alpha,  ddof = 1)/np.sqrt(len(fisher_pvals)))
        se_s.append(np.std(simes_pvals <= alpha,  ddof = 1)/np.sqrt(len(simes_pvals)))
        se_b.append(np.std(bon_pvals <= alpha,  ddof = 1)/np.sqrt(len(bon_pvals)))
    return ([avg_pow_cpch_f, avg_pow_cpch_s, avg_pow_cpch_b, avg_pow_f, avg_pow_s, avg_pow_b],[se_cpch_f, se_cpch_s, se_cpch_b,se_f, se_s, se_b])

In [7]:
XX = np.array(dmd[['ts.1', 'ts.2', 'ts.3', 'ts.4']])
XX_pvals = np.array(dmd[['pvalue.1', 'pvalue.2', 'pvalue.3', 'pvalue.4']])
#Vmm(XX, 4, 2, f_fisher, norm.pdf, norm.cdf, truncnorm.rvs, 1000)

In [8]:
power, ses = calc_avg_power(XX, XX_pvals, 4, 0.05)

/Users/Biyonka/Documents/Research/minmax/cpch/cpch_source.py:172: RuntimeWarning: divide by zero encountered in log
  return (-2 * np.sum(np.log(pvals), axis))


In [9]:
power_rounded = np.round(np.array(power)*100, 1)
print('r = 2 & ' + str(power_rounded[0][0]) + " & " + str(power_rounded[1][0]) + " & " + str(power_rounded[2][0]) + " & " + str(power_rounded[3][0])  + " & " + str(power_rounded[4][0])  + " & " + str(power_rounded[5][0])  + ' \\\\')
print('r = 3 & ' + str(power_rounded[0][1]) + " & " + str(power_rounded[1][1]) + " & " + str(power_rounded[2][1]) + " & " + str(power_rounded[3][1])  + " & " + str(power_rounded[4][1])  + " & " + str(power_rounded[5][1])    + ' \\\\')
#when r=m, all three combining functions will produce same p-values
print('r = 4 & ' + str(power_rounded[0][2]) + " & " + str(power_rounded[0][2]) + " & " + str(power_rounded[0][2]) + " & " + str(power_rounded[3][2])  + " & " + str(power_rounded[3][2])  + " & " + str(power_rounded[3][2])   + ' \\\\')

r = 2 & 27.8 & 24.9 & 24.7 & 23.4 & 20.1 & 19.7 \\
r = 3 & 18.3 & 17.7 & 16.2 & 12.8 & 11.7 & 11.6 \\
r = 4 & 9.2 & 9.2 & 9.2 & 5.0 & 5.0 & 5.0 \\


## Run Standard Multiple PCH Testing Analysis

In [57]:
def run_standard(data, alpha):
    pvals = ['pvalue.1', 'pvalue.2', 'pvalue.3', 'pvalue.4']
    ts = ['ts.1', 'ts.2', 'ts.3', 'ts.4']
    XX_pvals =  np.array(data[pvals])
    XX = np.array(data[ts])
    #store num rejections for each value of r
    cpch_fisher_storey, cpch_fisher_bh, cpch_simes_storey, cpch_simes_bh, cpch_bon_storey, cpch_bon_bh = [],[],[],[],[],[]
    fisher_storey, fisher_bh = [], []
    simes_storey, simes_bh = [], []
    bon_storey, bon_bh = [], []
    fisher_dhh, fisher_sdhh = [],[]
    simes_dhh, simes_sdhh = [],[]
    bon_dhh, bon_sdhh = [],[]
    adafilter = []
    for r in np.arange(2, 5, 1):
        print(r)
        #mm-based methods
        cpch_f = cpch(XX, 4, r, f_fisher, norm.pdf, norm.cdf, truncnorm.rvs, 100000)
        cpch_fisher_bh.append(sum(mt.multipletests(cpch_f, alpha, method = 'fdr_bh')[0]))
        cpch_fisher_storey.append(sum(storey(cpch_f, alpha)))
        print('mm_fisher_methods_done')
        #mm-based methods
        cpch_s = cpch(XX, 4, r, f_simes, norm.pdf, norm.cdf, truncnorm.rvs, 100000)
        cpch_simes_bh.append(sum(mt.multipletests(cpch_s, alpha, method = 'fdr_bh')[0]))
        cpch_simes_storey.append(sum(storey(cpch_s, alpha)))
        print('mm_simes_methods_done')
        #mm-based methods
        cpch_b = cpch(XX, 4, r, f_bonferroni, norm.pdf, norm.cdf, truncnorm.rvs, 100000)
        cpch_bon_bh.append(sum(mt.multipletests(cpch_b, alpha, method = 'fdr_bh')[0]))
        cpch_bon_storey.append(sum(storey(cpch_b, alpha)))
        print('mm_bon_methods_done')
        #fisher-based methods
        nr, nc = XX_pvals.shape
        Br = ro.r.matrix(XX_pvals, nrow=nr, ncol=nc)
        fisher_results = classical(Br, r = r, alpha = alpha, method = "Fisher")
        fisher_pvals = np.array(list(pd.DataFrame(fisher_results).to_numpy()[1, :]))
        simes_results = classical(Br, r = r, alpha = alpha, method = "Simes")
        simes_pvals = np.array(list(pd.DataFrame(fisher_results).to_numpy()[1, :]))
        bon_results = classical(Br, r = r, alpha = alpha, method = "Bonferroni")
        bon_pvals = np.array(list(pd.DataFrame(fisher_results).to_numpy()[1, :]))
        
        f_tau = fisher_pvals[fisher_pvals <= 0.1]
        dhh_f_pvals = f_tau/0.1
        fisher_dhh.append(sum(mt.multipletests(dhh_f_pvals, alpha, method = 'fdr_bh')[0]))
        fisher_sdhh.append(sum(storey(dhh_f_pvals, 0.1)))
        
        s_tau = simes_pvals[simes_pvals <= 0.1]
        dhh_s_pvals = s_tau/0.1
        simes_dhh.append(sum(mt.multipletests(dhh_s_pvals, alpha, method = 'fdr_bh')[0]))
        simes_sdhh.append(sum(storey(dhh_s_pvals, 0.1)))
        
        b_tau = bon_pvals[bon_pvals <= 0.1]
        dhh_b_pvals = b_tau/0.1
        bon_dhh.append(sum(mt.multipletests(dhh_b_pvals, alpha, method = 'fdr_bh')[0]))
        bon_sdhh.append(sum(storey(dhh_b_pvals, 0.1)))   
        
        fisher_bh.append(sum(mt.multipletests(fisher_pvals, alpha, method = 'fdr_bh')[0]))
        fisher_storey.append(sum(storey(fisher_pvals, alpha)))
        simes_bh.append(sum(mt.multipletests(simes_pvals, alpha, method = 'fdr_bh')[0]))
        simes_storey.append(sum(storey(simes_pvals, alpha)))
        bon_bh.append(sum(mt.multipletests(bon_pvals, alpha, method = 'fdr_bh')[0]))
        bon_storey.append(sum(storey(bon_pvals, alpha)))

        
        #adafilter
        ada_results = af(Br, r = r, alpha = alpha)
        ada_decision = pd.DataFrame(ada_results).to_numpy()[:,0]
        adafilter.append(sum(ada_decision))
        
    rejections = [cpch_fisher_storey, cpch_fisher_bh, fisher_storey, fisher_bh, 
                  cpch_simes_storey, cpch_simes_bh, simes_storey, simes_bh,
                  cpch_bon_storey, cpch_bon_bh, bon_storey, bon_bh,
                  fisher_dhh, fisher_sdhh, simes_dhh, simes_sdhh, bon_dhh, bon_sdhh,
                  adafilter]
    print("cPCH-Fisher-Storey & " + str(rejections[0][0]) + " & " + str(rejections[0][1]) + " & " + str(rejections[0][2]) +' \\\\')
    print("cPCH-Fisher-BH & " + str(rejections[1][0]) + " & " + str(rejections[1][1]) + " & " + str(rejections[1][2]) + ' \\\\')
    print("Fisher-Storey & " + str(rejections[2][0]) + " & " + str(rejections[2][1]) +" & " + str(rejections[2][2]) + ' \\\\')
    print("Fisher-BH & " + str(rejections[3][0]) + " & " + str(rejections[3][1]) + " & " + str(rejections[3][2]) +' \\\\')
    print("cPCH-Simes-Storey & " + str(rejections[4][0]) + " & " + str(rejections[4][1]) +" & " + str(rejections[4][2]) + ' \\\\')
    print("cPCH-Simes-BH & " + str(rejections[5][0]) + " & " + str(rejections[5][1]) + " & " + str(rejections[5][2]) +' \\\\')
    print("Simes-Storey & " + str(rejections[6][0]) + " & " + str(rejections[6][1]) + " & " + str(rejections[6][2]) +' \\\\')
    print("Simes-BH & " + str(rejections[7][0]) + " & " + str(rejections[7][1]) + " & " + str(rejections[7][2]) +' \\\\')
    print("cPCH-Bonferroni-Storey & " + str(rejections[8][0]) + " & " + str(rejections[8][1]) +" & " + str(rejections[8][2]) + ' \\\\')
    print("cPCH-Bonferroni-BH & " + str(rejections[9][0]) + " & " + str(rejections[9][1]) + " & " + str(rejections[9][2]) +' \\\\')
    print("Bonferroni-Storey & " + str(rejections[10][0]) + " & " + str(rejections[10][1]) + " & " + str(rejections[10][2]) +' \\\\')
    print("Bonferroni-BH & " + str(rejections[11][0]) + " & " + str(rejections[11][1]) + " & " + str(rejections[11][2]) +' \\\\')
    print("Fisher-DHH & " + str(rejections[12][0]) + " & " + str(rejections[12][1]) +  " & " + str(rejections[12][2]) + ' \\\\')
    print("Fisher-sDHH & " + str(rejections[13][0]) + " & " + str(rejections[13][1]) +  " & " + str(rejections[13][2]) + ' \\\\')
    print("Simes-DHH & " + str(rejections[14][0]) + " & " + str(rejections[14][1]) +  " & " + str(rejections[14][2]) + ' \\\\')
    print("Simes-sDHH & " + str(rejections[15][0]) + " & " + str(rejections[15][1]) +  " & " + str(rejections[15][2]) + ' \\\\')
    print("Bonferroni-DHH & " + str(rejections[16][0]) + " & " + str(rejections[16][1]) +  " & " + str(rejections[16][2]) + ' \\\\')
    print("Bonferroni-sDHH & " + str(rejections[17][0]) + " & " + str(rejections[17][1]) +  " & " + str(rejections[17][2]) + ' \\\\')
    print("adaFilter & " + str(rejections[18][0]) + " & " + str(rejections[18][1]) + " & " + str(rejections[18][2]) +' \\\\')

    return(rejections)

In [58]:
run_standard(dmd, 0.1)

2
mm_fisher_methods_done
mm_simes_methods_done
mm_bon_methods_done
3
mm_fisher_methods_done
mm_simes_methods_done
mm_bon_methods_done
4
mm_fisher_methods_done
mm_simes_methods_done
mm_bon_methods_done
cPCH-Fisher-Storey & 497 & 193 & 14 \\
cPCH-Fisher-BH & 425 & 167 & 13 \\
Fisher-Storey & 364 & 128 & 9 \\
Fisher-BH & 359 & 128 & 9 \\
cPCH-Simes-Storey & 409 & 165 & 13 \\
cPCH-Simes-BH & 340 & 114 & 12 \\
Simes-Storey & 364 & 128 & 9 \\
Simes-BH & 359 & 128 & 9 \\
cPCH-Bonferroni-Storey & 400 & 141 & 15 \\
cPCH-Bonferroni-BH & 336 & 109 & 13 \\
Bonferroni-Storey & 364 & 128 & 9 \\
Bonferroni-BH & 359 & 128 & 9 \\
Fisher-DHH & 284 & 115 & 9 \\
Fisher-sDHH & 382 & 151 & 21 \\
Simes-DHH & 284 & 115 & 9 \\
Simes-sDHH & 382 & 151 & 21 \\
Bonferroni-DHH & 284 & 115 & 9 \\
Bonferroni-sDHH & 382 & 151 & 21 \\
adaFilter & 380.0 & 217.0 & 73.0 \\


[[497, 193, 14],
 [425, 167, 13],
 [364, 128, 9],
 [359, 128, 9],
 [409, 165, 13],
 [340, 114, 12],
 [364, 128, 9],
 [359, 128, 9],
 [400, 141, 15],
 [336, 109, 13],
 [364, 128, 9],
 [359, 128, 9],
 [284, 115, 9],
 [382, 151, 21],
 [284, 115, 9],
 [382, 151, 21],
 [284, 115, 9],
 [382, 151, 21],
 [380.0, 217.0, 73.0]]